<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/3_dise%C3%B1o_entrenamiento_evaluacion/3_2_Generacion_X_y.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2. Generación de X e y para modelos

## 0. Clonado de Repositorio, instalación de librería e importación.

### Clonado de Repositorio

In [2]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

Cloning into 'neural_profit'...
remote: Enumerating objects: 370, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 370 (delta 56), reused 15 (delta 8), pack-reused 279 (from 2)
Receiving objects: 100% (370/370), 246.57 MiB | 16.75 MiB/s, done.
Resolving deltas: 100% (206/206), done.
Updating files: 100% (60/60), done.


### Acceso de Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### Instalación de librerías

In [7]:
# !{sys.executable} -m pip install -q pandas_market_calendars  # Solo si usás horarios de mercados
!{sys.executable} -m pip install -q ta
print("Librerías instaladas: ta")

  Preparing metadata (setup.py) ... done
Librerías instaladas: ta


### Importación de librerías

In [8]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Utilidades del sistema y fechas
import os
import glob
import requests
from datetime import datetime, timedelta

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Análisis técnico
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

# Estadística
from scipy.stats import spearmanr

#que es?
#from tqdm.notebook import tqdm
from tqdm import tqdm
# Modelos ML
#from xgboost import XGBRegressor
#from sklearn.metrics import mean_squared_error, r2_score

# Calendario de mercados (descomentar si lo necesitás)
# import pandas_market_calendars as mcal

## 1. Carga de datasets train, valid y test.

In [9]:
def load_data_from_drive():
    """
    Función para cargar un archivo Parquet desde el drive
        """
    # Definir la URL del archivo Parquet en Drive
    df_path_mnq = f'{drive_path}/mnq_data/mnq_model.parquet'
    df_path_train = f'{drive_path}/mnq_data/mnq_train.parquet'
    df_path_valid = f'{drive_path}/mnq_data/mnq_valid.parquet'
    df_path_test = f'{drive_path}/mnq_data/mnq_test.parquet'
    df_path_factores = f'{drive_path}/df_factores.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [10]:
def load_data_from_repo():
    """
    Función para cargar un archivo Parquet desde el repositorio clonado
    """
    # Definir la URL del archivo Parquet en GitHub
    df_path_mnq = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_model.parquet'
    df_path_factores = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/df_factores.parquet'
    df_path_train = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_train.parquet'
    df_path_valid = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_valid.parquet'
    df_path_test = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_test.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [11]:
mnq_model, indicadores_tecnicos, mnq_train, mnq_valid, mnq_test = load_data_from_drive()

## 1.1. Información de los datasets

In [12]:
def info_dataset (df): # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"Cantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['target_return_30']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"Valores por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo
  print(f"Hora diaria de inicio {primer_hora}")
  print(f"Hora diaria de final {ultima_hora}")
  print(f"Zona horaria: {zona_horaria}")

In [13]:
info_dataset(mnq_train)

Cantidad de días: 917
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [14]:
info_dataset(mnq_valid)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [15]:
info_dataset(mnq_test)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


## 2. Generación de ventanas X e y

In [16]:
target_column = "target_return_30"
features = mnq_model.columns.tolist()
features.remove(target_column)
features.remove('date')

window_size = 60

In [17]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue
            vector = ventana.values.flatten()
            target = grupo.loc[i+window_size-1, target_col]
            X.append(vector)
            y.append(target)
    return np.array(X), np.array(y)

### Generamos los X_* e y_* con el total de features

Antes de correr la generación de X e y, tenemos que verificar si es que no existe en la carpeta ventanas_X_y:


In [18]:
# Subcarpeta donde querés guardar
save_dir = f"{drive_path}/ventanas_x_y"

# Crear carpeta si no existe
os.makedirs(save_dir, exist_ok=True)


In [19]:
#Ruta de x_y
ruta_x_y_train = f"{drive_path}/ventanas_x_y/mnq_Xy_train.npz"
ruta_x_y_valid = f"{drive_path}/ventanas_x_y/mnq_Xy_valid.npz"
ruta_x_y_test = f"{drive_path}/ventanas_x_y/mnq_Xy_test.npz"


#### Para X_train e y_train

In [20]:
if not os.path.exists(ruta_x_y_train):
    print('El archivo no existe -> Generando X_train e y_train: ')
    X_train, y_train = generar_ventanas(mnq_train, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_train, X=X_train, y=y_train)
    print("Guardado:", ruta_x_y_train)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_train)
    data_train = np.load(ruta_x_y_train)
    X_train, y_train = data_train["X"], data_train["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_train.npz


In [21]:
print(f"X_train info: {X_train.shape[0]} ventanas aplanadas en {X_train.shape[1]} números, el equivalente a la window size ({window_size}) por la cantidad de features ({len(features)})")
print(f"y_train info: {y_train.shape[0]} valores que corresponden al retorno a 30 minutos ({target_column})")

X_train info: 276017 ventanas aplanadas en 1260 números, el equivalente a la window size (60) por la cantidad de features (21)
y_train info: 276017 valores que corresponden al retorno a 30 minutos (target_return_30)


#### Para X_valid e y_valid

In [22]:
if not os.path.exists(ruta_x_y_valid):
    print('El archivo no existe -> Generando X_valid e y_valid: ')
    X_valid, y_valid = generar_ventanas(mnq_valid, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_valid, X=X_valid, y=y_valid)
    print("Guardado:", ruta_x_y_valid)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_valid)
    data_valid = np.load(ruta_x_y_valid)
    X_valid, y_valid = data_valid["X"], data_valid["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_valid.npz


#### Para X_test e y_test

In [23]:
if not os.path.exists(ruta_x_y_test):
    print('El archivo no existe -> Generando X_test e y_test: ')
    X_test, y_test = generar_ventanas(mnq_test, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_test, X=X_test, y=y_test)
    print("Guardado:", ruta_x_y_test)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_test)
    data_test = np.load(ruta_x_y_test)
    X_test, y_test = data_test["X"], data_test["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_test.npz


## 3. Selección y filtrado de features

### 3.1. Funciones

#### Función para encontrar los índices de los features seleccionados

In [56]:
def get_feature_indices(tabla_features, feature_names):
    """
    Devuelve los índices de los features dados sus nombres.

    Parámetros:
    -----------
    tabla_features : pd.DataFrame
        DataFrame con columnas ["Índice", "Feature"].
    feature_names : list[str]
        Lista de nombres de features a buscar.

    Retorna:
    --------
    list[int] : índices correspondientes a los nombres.
    """
    idx_list = [0, 1, 2, 3, 4] #OHCLV incluido
    for name in feature_names:
            fila = tabla_features.index[tabla_features["Feature"] == name]
            if len(fila) > 0:
                idx_list.append(int(fila[0]))
            else:
                print(f"⚠️ Feature '{name}' no encontrado en tabla_features.")

        # Ordenar y eliminar duplicados
    idx_list = sorted(set(idx_list))
    return idx_list

#### Función para filtrar features de los X_*

In [57]:
def filter_features_to_model(X_train, X_valid, X_test, features, keep_idx, window_size=60):
    """
    Filtra features de los conjuntos X_* manteniendo solo los índices indicados.

    Parámetros:
    -----------
    X_train, X_valid, X_test : np.ndarray
        Arrays en 2D (n_muestras, window_size * n_features) aplanados.
    features : list[str]
        Lista completa de nombres de features en el orden original.
    window_size : int
        Tamaño de la ventana usada en la construcción (ej: 60).
    keep_idx : list[int]
        Lista de índices de features que se quieren conservar (ej: [0,1,2,3,4,7]).

    Retorna:
    --------
    X_train_f, X_valid_f, X_test_f : np.ndarray
        Arrays filtrados en 2D (n_muestras, window_size * n_features_seleccionados).
    features_f : list[str]
        Lista de features seleccionados.
    """

    n_features = len(features)

    # Verificación rápida
    n_total_cols = window_size * n_features
    assert X_train.shape[1] == n_total_cols, "X_train no coincide con window_size * n_features"

    # Calcular columnas a mantener
    cols_to_keep = []
    for idx in keep_idx:
        start = idx * window_size
        end = (idx + 1) * window_size
        cols_to_keep.extend(range(start, end))

    # Filtrar arrays
    X_train_f = X_train[:, cols_to_keep]
    X_valid_f = X_valid[:, cols_to_keep]
    X_test_f  = X_test[:, cols_to_keep]

    # Features filtrados
    features_f = [features[i] for i in keep_idx]

    print(f"Features seleccionados ({len(features_f)}): {features_f}")
    print("X_train_f shape:", X_train_f.shape)
    print("X_valid_f shape:", X_valid_f.shape)
    print("X_test_f shape :", X_test_f.shape)

    return X_train_f, X_valid_f, X_test_f, features_f

#### Tabla de features

In [58]:
# Crear DataFrame con índice y nombre del feature
tabla_features = pd.DataFrame({"Feature": features})
tabla_features

,Feature
0,open
1,high
2,low
3,close
4,volume
5,momentum_3
6,momentum_10
7,roc_5
8,roc_20
9,rsi_3


#### Función de aplicación

In [59]:
#Filtrar
def application_filter (features_to_model):
    X_train_f, X_valid_f, X_test_f, features_f = filter_features_to_model(
        X_train, X_valid, X_test,
        features=features,          # tu lista de 21 features
        window_size=60,
        keep_idx= get_feature_indices(tabla_features, features_to_model)
    )
    return X_train_f, X_valid_f, X_test_f, features_f


### 3.2. Aplicación de filtrado

In [54]:
 features_to_model = ['factor30', 'rsi_14', 'price_ema30', 'stoch_k_20', 'bb_percent_30_20', 'reversal_momentum_factor', 'rsi_7', 'reversal_media_factor', 'rsi_3', 'bb_percent_20_15']

In [55]:
X_train_f, X_valid_f, X_test_f, features_f = application_filter(features_to_model)

Features seleccionados (15): ['open', 'high', 'low', 'close', 'volume', 'rsi_3', 'rsi_7', 'rsi_14', 'stoch_k_20', 'bb_percent_20_15', 'bb_percent_30_20', 'price_ema30', 'reversal_momentum_factor', 'reversal_media_factor', 'factor30']
X_train_f shape: (276017, 900)
X_valid_f shape: (59297, 900)
X_test_f shape : (59297, 900)
